# Side-by-side comparison videos: CHOMP cold vs. Model vs. CHOMP warm

For a chosen dataset sample (by index) this notebook renders one video with three panels:

| left | middle | right |
|------|--------|-------|
| **CHOMP cold start** (optimized from a straight line) | **Model** (WarmStartPlanner forward pass) | **CHOMP warm start** (CHOMP refining the model output) |

Two videos are produced: one on the **circle-obstacle** dataset and one on the **voxel-shape** dataset.
Set `INDEX` in the run cells to pick which sample to animate. Videos are written to `resources/videos/`.

Run with the project venv kernel (`.venv`, Python 3.11.15).

In [1]:
# ---- Setup ----
import sys, os, io, contextlib, shutil, subprocess, tempfile
import numpy as np
import torch
import plotly.graph_objects as go
from PIL import Image, ImageDraw, ImageFont

# Run from src/ so the relative data/ and models/ paths resolve like the other notebooks.
if os.path.basename(os.getcwd()) != "src":
    os.chdir(os.path.join(os.getcwd(), "src"))

SIMPLEARM_PATH = os.path.abspath("../external/SimpleArm/src")
if SIMPLEARM_PATH not in sys.path:
    sys.path.insert(0, SIMPLEARM_PATH)

from simplearm.robot import RobotInfo
from simplearm.geom import Obstacles, SquareGrid
from simplearm.viz import RobotViewer

from chomp import CHOMPOptimizer
from evaluation import TrajectoryEvaluator
import models

DEVICE = "cpu"
torch.manual_seed(0)
np.random.seed(0)

# Panel order (left -> right) and per-method display style, matching chomp_test.ipynb colours.
ORDER  = ["CHOMP (cold)", "Network", "CHOMP (warm)"]
LABELS = {"CHOMP (cold)": "CHOMP cold start", "Network": "Model", "CHOMP (warm)": "CHOMP warm start"}
COLORS = {"CHOMP (cold)": "#eb6834", "Network": "#2a78d6", "CHOMP (warm)": "#1baf7a"}
print("setup done | cwd:", os.getcwd())

setup done | cwd: /Users/timomatuszewski/Desktop/SS26/Advanced_Deep_Learning_for_Robotics/Self-Supervised-Learning-for-Robot-Motion-Planning/src


In [2]:
# ---- Pipeline: load a dataset + model and build the CHOMP optimizer ----
def build_pipeline(name, dataset_path, model_path, train_loss, eta_scratch, eta_warm,
                   T=50, C=10, device=DEVICE):
    """Load dataset + model and construct the CHOMP optimizer straight from the dataset
    metadata (same setup as chomp_test.ipynb). Returns a `pipe` dict used by everything below."""
    ds   = torch.load(dataset_path, weights_only=False)
    meta = ds["metadata"]

    # CHOMP is pointed at the model's own training loss weights, so no method is handicapped.
    chomp = CHOMPOptimizer.from_metadata(meta, T=T, device=device, **train_loss)
    ev    = TrajectoryEvaluator.from_metadata(meta, device=device)

    model = models.WarmStartPlanner(dof=meta["dof"], T=T, C=C,
                                    linklengths=meta["linklengths"]).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model.eval()

    print(f"[{name}] N={meta['N']} samples | dof={meta['dof']} | "
          f"world={'voxels' if 'voxels' in ds else 'circles'}")
    return dict(name=name, ds=ds, meta=meta, chomp=chomp, ev=ev, robot=chomp.robot,
                model=model, device=device, T=T, C=C,
                eta_scratch=eta_scratch, eta_warm=eta_warm)


def build_world(pipe, i):
    """Drawable world geometry for sample `i` (voxel SquareGrid for shape datasets, circle
    Obstacles otherwise), matching how the dataset was built. Same logic as chomp_test.ipynb."""
    ds, meta = pipe["ds"], pipe["meta"]
    if "voxels" in ds:
        vox = ds["voxels"][i, 0].numpy().astype(bool)
        return SquareGrid.from_zero_centered(
            limits=(-meta["grid_length"] / 2, meta["grid_length"] / 2), data=vox.T)
    n_obs = ds["n_obstacles"][i].item()
    obs = ds["obstacles"][i, :n_obs]
    return Obstacles(x=obs[:, 0].numpy(), y=obs[:, 1].numpy(), r=obs[:, 2].numpy())


def compute_trajectories(pipe, i):
    """Run the three methods on sample `i`: model forward pass, CHOMP from a straight line
    (cold), and CHOMP warm-started from the model output. Returns trajectories, the world,
    and each method's collision-free flag."""
    ds, model, chomp, ev = pipe["ds"], pipe["model"], pipe["chomp"], pipe["ev"]
    sdf_i = ds["sdf"][i]
    qs, qg = ds["q_start"][i:i+1], ds["q_goal"][i:i+1]

    with torch.no_grad():
        wp = model(qs, qg, sdf_i.unsqueeze(0))
        traj_model = model.trajectory(wp)
    traj_cold = chomp.optimize(sdf_i, qs, qg, max_iters=500, eta=pipe["eta_scratch"])
    traj_warm = chomp.optimize(sdf_i, qs, qg, init_waypoints=wp, max_iters=500, eta=pipe["eta_warm"])

    trajs = {"Network": traj_model, "CHOMP (cold)": traj_cold, "CHOMP (warm)": traj_warm}
    status = {k: bool(ev.evaluate(v, sdf_i)["collision_free"]) for k, v in trajs.items()}
    return trajs, build_world(pipe, i), status

In [3]:
# ---- Rendering: trajectory -> per-frame PNGs -> labelled 3-panel video ----
def _viewer_frames(traj, robot, world, size=420):
    """Render an animated RobotViewer to one clean PIL image per timestep (no play/slider UI,
    no axis ticks). Preserves the voxel background image for shape worlds."""
    viz = RobotViewer(traj.squeeze(0).cpu().numpy(), robot, obstacles=world, animate=True)
    _orig = go.Figure.show
    go.Figure.show = lambda *a, **k: None  # RobotViewer.plot() calls fig.show(); mute it.
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            viz.plot()
    finally:
        go.Figure.show = _orig

    imgs = []
    for pf in viz.fig.frames:
        tmp = go.Figure(data=pf.data, layout=viz.fig.layout)
        tmp.layout.updatemenus = ()   # drop Play/Pause buttons
        tmp.layout.sliders = ()       # drop the frame slider
        tmp.update_layout(margin=dict(l=4, r=4, t=4, b=4), width=size, height=size,
                          showlegend=False, paper_bgcolor="white", plot_bgcolor="white")
        tmp.update_xaxes(showticklabels=False, showgrid=False, zeroline=False, ticks="")
        tmp.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, ticks="")
        png = tmp.to_image(format="png", width=size, height=size)
        imgs.append(Image.open(io.BytesIO(png)).convert("RGB"))
    return imgs


def _load_fonts():
    """A truetype font if available (macOS Helvetica), else Pillow's bitmap default."""
    for path in ("/System/Library/Fonts/Helvetica.ttc", "/System/Library/Fonts/Supplemental/Arial.ttf"):
        if os.path.exists(path):
            return ImageFont.truetype(path, 22), ImageFont.truetype(path, 15)
    d = ImageFont.load_default()
    return d, d


def _encode_mp4(frames, out_path, fps):
    """Encode PIL frames to an MP4 via the system ffmpeg (imageio's mp4 writer is unavailable
    here). Falls back to a GIF next to `out_path` if ffmpeg is missing."""
    if shutil.which("ffmpeg") is None:
        gif = os.path.splitext(out_path)[0] + ".gif"
        _save_gif(frames, gif, fps)
        return gif
    with tempfile.TemporaryDirectory() as td:
        for k, im in enumerate(frames):
            im.save(os.path.join(td, f"f{k:04d}.png"))
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(fps),
               "-i", os.path.join(td, "f%04d.png"), "-pix_fmt", "yuv420p",
               "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", out_path]
        subprocess.run(cmd, check=True)
    return out_path


def _save_gif(frames, out_path, fps):
    frames[0].save(out_path, save_all=True, append_images=frames[1:], loop=0,
                   duration=int(1000 / fps), optimize=False)
    return out_path


def make_comparison_video(pipe, i, out_dir="resources/videos", fmt="mp4", fps=12, size=420):
    """Build the labelled 3-panel (CHOMP cold | Model | CHOMP warm) video for sample `i`
    and save it to `out_dir`. Returns the output path."""
    trajs, world, status = compute_trajectories(pipe, i)
    print(f"[{pipe['name']}] sample {i} rendering frames ...")
    panels = [(k, _viewer_frames(trajs[k], pipe["robot"], world, size=size)) for k in ORDER]

    n_frames = len(panels[0][1])
    header   = 48
    W, H     = size * 3, size + header
    title_f, sub_f = _load_fonts()

    composed = []
    for t in range(n_frames):
        canvas = Image.new("RGB", (W, H), "white")
        draw   = ImageDraw.Draw(canvas)
        for p, (key, fr) in enumerate(panels):
            x0 = p * size
            canvas.paste(fr[t], (x0, header))
            cx = x0 + size // 2
            draw.text((cx, 14), LABELS[key], fill=COLORS[key], font=title_f, anchor="mm")
            free = status[key]
            draw.text((cx, 35), "collision-free" if free else "in collision",
                      fill="#1baf7a" if free else "#d1495b", font=sub_f, anchor="mm")
            if p > 0:  # thin separator between panels
                draw.line([(x0, header), (x0, H)], fill="#e1e0d9", width=1)
        composed.append(canvas)

    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, f"{pipe['name']}_compare_sample_{i}.{fmt}")
    out = _encode_mp4(composed, out, fps) if fmt == "mp4" else _save_gif(composed, out, fps)
    print(f"[{pipe['name']}] saved video: {out}  ({n_frames} frames, {fps} fps)")
    return out

## Video 1 — circle obstacles

Dataset `dataset_general_0807_test` with model `wsp_general_0907_v1`. Change `INDEX` to animate a
different sample (0 .. N-1).

In [4]:
pipe_circles = build_pipeline(
    name         = "circles",
    dataset_path = "data/dataset_general_0807_test.pt",
    model_path   = "models/wsp_general_0907_v1.pt",
    train_loss   = dict(eps=0.15, w_coll=350.0, w_joints=1.0, w_smooth=1.0, collision_agg="sum"),
    eta_scratch  = 300,
    eta_warm     = 1500,
)

INDEX = 46   # <-- pick the dataset sample to animate
for i in range(INDEX, INDEX + 50):
    make_comparison_video(pipe_circles, i, fmt="mp4", fps=12, size=420)

[circles] N=1501 samples | dof=3 | world=circles
[circles] sample 46 rendering frames ...


/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

⠋ Plotting robot... (0:00:00.00)

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)


/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

...

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

⠋ Plotting robot... (0:00:00.00)

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)


/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_98160/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

[circles] saved video: resources/videos/circles_compare_sample_95.mp4  (50 frames, 12 fps)


## Video 2 — voxel shapes

Dataset `dataset_general_shapes_1707_test` with model `wsp_general_shapes_1707_v1`. Change `INDEX`
to animate a different sample.

In [16]:
pipe_shapes = build_pipeline(
    name         = "shapes",
    dataset_path = "data/dataset_general_shapes_1707_test.pt",
    model_path   = "models/wsp_general_shapes_1707_v1.pt",
    train_loss   = dict(eps=0.15, w_coll=400.0, w_joints=1.0, w_smooth=1.0, collision_agg="sum"),
    eta_scratch  = 300,
    eta_warm     = 1500,
)

INDEX = 25   # <-- pick the dataset sample to animate
for i in range(INDEX, INDEX + 20):
    make_comparison_video(pipe_shapes, i, fmt="mp4", fps=12, size=420)

[shapes] N=1500 samples | dof=3 | world=voxels
[shapes] sample 25 rendering frames ...


/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

[shapes] saved video: resources/videos/shapes_compare_sample_25.mp4  (50 frames, 12 fps)
[shapes] sample 26 rendering frames ...


...

[shapes] saved video: resources/videos/shapes_compare_sample_43.mp4  (50 frames, 12 fps)
[shapes] sample 44 rendering frames ...


/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  png = tmp.to_image(format="png", width=size, height=size)
/var/folders/80/7rcx9l0s05z64ktvn81xcq580000gn/T/ipykernel_87294/2438201158.py:23: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido 

[shapes] saved video: resources/videos/shapes_compare_sample_44.mp4  (50 frames, 12 fps)
